# 计算机图形学实验六：可微渲染

本笔记本以 `cow.obj` 作为目标网格，使用 PyTorch3D 的软剪影渲染器，在 4 个视角下对一个细分球体进行 300 次可微优化。

运行前：在 Colab 中选择 **运行时 → 更改运行时类型 → T4 GPU**，然后按顺序运行全部单元格。

In [ ]:
# 安装 PyTorch3D。
# 逻辑与 PyTorch3D 官方教程一致：torch 2.2/Linux 优先使用匹配的预编译 wheel，
# 其他情况从官方 stable 分支构建。首次运行可能需要几分钟。
import os
import sys
import subprocess
import torch

try:
    import pytorch3d
    print("PyTorch3D 已安装。")
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fvcore", "iopath", "imageio"])

    if torch.__version__.startswith("2.2.") and sys.platform.startswith("linux"):
        pyt_version_str = torch.__version__.split("+")[0].replace(".", "")
        version_str = "".join([
            f"py3{sys.version_info.minor}_cu",
            torch.version.cuda.replace(".", ""),
            f"_pyt{pyt_version_str}",
        ])
        wheel_url = (
            "https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/"
            f"{version_str}/download.html"
        )
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--no-index", "--no-cache-dir",
            "pytorch3d", "-f", wheel_url,
        ])
    else:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/facebookresearch/pytorch3d.git@stable",
        ])
    import pytorch3d

print("PyTorch:", torch.__version__)
print("PyTorch3D:", getattr(pytorch3d, "__version__", "installed"))
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → T4 GPU，然后重新运行。")


In [ ]:
# 上传目标奶牛模型。请选择本项目中的 cow.obj。
from pathlib import Path
import shutil
import os

if not Path("cow.obj").exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("没有上传模型文件。")
    uploaded_name = next(iter(uploaded))
    shutil.copy2(uploaded_name, "cow.obj")

print("当前目标模型：", Path("cow.obj").resolve())
print("文件大小：", Path("cow.obj").stat().st_size, "bytes")

In [ ]:
# 导入库与通用工具
import os
import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import torch

from pytorch3d.io import load_objs_as_meshes, save_obj
from pytorch3d.renderer import (
    BlendParams,
    FoVPerspectiveCameras,
    MeshRasterizer,
    MeshRenderer,
    RasterizationSettings,
    SoftSilhouetteShader,
    look_at_view_transform,
)
from pytorch3d.structures import Meshes
from pytorch3d.utils import ico_sphere
from pytorch3d.loss import (
    mesh_edge_loss,
    mesh_laplacian_smoothing,
    mesh_normal_consistency,
)

device = torch.device("cuda:0")
torch.manual_seed(42)
np.random.seed(42)

RESULTS_DIR = Path("work06_results")
RESULTS_DIR.mkdir(exist_ok=True)

NUM_VIEWS = 4
IMAGE_SIZE = 128
NUM_ITERS = 300
SAVE_EVERY = 20

def normalize_mesh(mesh: Meshes) -> Meshes:
    """将目标网格平移至原点附近并缩放到统一尺度。"""
    verts = mesh.verts_packed()
    center = verts.mean(0)
    scale = (verts - center).abs().max()
    return mesh.offset_verts(-center).scale_verts(1.0 / scale)

def silhouette_grid(silhouettes: torch.Tensor) -> np.ndarray:
    """把 N 张单通道剪影图横向拼接为 RGB 图像。"""
    arr = silhouettes.detach().float().cpu().numpy()
    arr = np.clip(arr, 0.0, 1.0)
    grid = np.concatenate([arr[i] for i in range(arr.shape[0])], axis=1)
    grid = (grid * 255).astype(np.uint8)
    return np.repeat(grid[..., None], 3, axis=2)

def save_single_row(silhouettes: torch.Tensor, title: str, out_path: Path):
    fig, axes = plt.subplots(1, silhouettes.shape[0], figsize=(4 * silhouettes.shape[0], 4))
    if silhouettes.shape[0] == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(silhouettes[i].detach().cpu(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"View {i + 1}")
        ax.axis("off")
    fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [ ]:
# 读取目标奶牛网格，设置多视角相机，并生成目标剪影
target_mesh = load_objs_as_meshes(["cow.obj"], device=device)
target_mesh = normalize_mesh(target_mesh)

# 在 4 个环绕视角下观测目标模型
elev = torch.tensor([10.0, 10.0, 10.0, 10.0], device=device)
azim = torch.tensor([0.0, 90.0, 180.0, 270.0], device=device)
R, T = look_at_view_transform(dist=3.0, elev=elev, azim=azim, device=device)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

# 软光栅化设置：边界处保留连续梯度，便于顶点优化
blend_params = BlendParams(sigma=1e-4, gamma=1e-4)
raster_settings = RasterizationSettings(
    image_size=IMAGE_SIZE,
    blur_radius=np.log(1.0 / 1e-4 - 1.0) * blend_params.sigma,
    faces_per_pixel=25,
    bin_size=None,
)
silhouette_renderer = MeshRenderer(
    rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
    shader=SoftSilhouetteShader(blend_params=blend_params),
)

with torch.no_grad():
    target_silhouettes = silhouette_renderer(
        target_mesh.extend(NUM_VIEWS), cameras=cameras
    )[..., 3]

save_single_row(
    target_silhouettes,
    "Target Cow Silhouettes",
    RESULTS_DIR / "target_silhouettes.png",
)

# 源网格：高细分球体。真正被优化的是 deform_verts（顶点偏移量）。
source_mesh = ico_sphere(level=4, device=device).scale_verts(0.85)
deform_verts = torch.zeros_like(source_mesh.verts_packed(), requires_grad=True)

print("目标网格：", target_mesh.verts_packed().shape[0], "vertices,",
      target_mesh.faces_packed().shape[0], "faces")
print("源球体：", source_mesh.verts_packed().shape[0], "vertices,",
      source_mesh.faces_packed().shape[0], "faces")

# 先保存初始球体剪影，便于与最终结果比较
with torch.no_grad():
    initial_silhouettes = silhouette_renderer(
        source_mesh.extend(NUM_VIEWS), cameras=cameras
    )[..., 3]
save_single_row(
    initial_silhouettes,
    "Initial Sphere Silhouettes",
    RESULTS_DIR / "initial_sphere_silhouettes.png",
)

In [ ]:
# 执行 300 次可微优化
optimizer = torch.optim.Adam([deform_verts], lr=0.08)

# 正则化权重：保证形状逼近剪影的同时，避免网格严重拉伸、尖刺或面片翻折。
W_LAP = 0.02
W_EDGE = 0.005
W_NORMAL = 0.002

history_total = []
history_silhouette = []
gif_frames = []

for epoch in range(NUM_ITERS):
    optimizer.zero_grad()

    # 根据当前顶点偏移量构造新的源网格
    current_mesh = source_mesh.offset_verts(deform_verts)

    # 在全部视角下渲染当前软剪影
    predicted_silhouettes = silhouette_renderer(
        current_mesh.extend(NUM_VIEWS), cameras=cameras
    )[..., 3]

    # 剪影误差 + 网格正则化
    loss_silhouette = torch.mean((predicted_silhouettes - target_silhouettes) ** 2)
    loss_lap = mesh_laplacian_smoothing(current_mesh, method="uniform")
    loss_edge = mesh_edge_loss(current_mesh)
    loss_normal = mesh_normal_consistency(current_mesh)

    total_loss = (
        loss_silhouette
        + W_LAP * loss_lap
        + W_EDGE * loss_edge
        + W_NORMAL * loss_normal
    )

    total_loss.backward()
    optimizer.step()

    history_total.append(float(total_loss.detach().cpu()))
    history_silhouette.append(float(loss_silhouette.detach().cpu()))

    # 每 20 次保存一次当前剪影，用于生成 GIF
    if epoch % SAVE_EVERY == 0 or epoch == NUM_ITERS - 1:
        with torch.no_grad():
            frame = silhouette_grid(predicted_silhouettes)
        gif_frames.append(frame)
        print(
            f"Epoch {epoch + 1:03d}/{NUM_ITERS} | "
            f"total={total_loss.item():.6f} | "
            f"silhouette={loss_silhouette.item():.6f}"
        )

# 最终网格与最终剪影
final_mesh = source_mesh.offset_verts(deform_verts.detach())
with torch.no_grad():
    final_silhouettes = silhouette_renderer(
        final_mesh.extend(NUM_VIEWS), cameras=cameras
    )[..., 3]

print("优化完成。最终总损失：", history_total[-1])
print("最终剪影 MSE：", history_silhouette[-1])

In [ ]:
# 保存实验结果：对比图、损失曲线、优化 GIF、最终 OBJ 和指标文件

# 1) 目标剪影与最终剪影对比
fig, axes = plt.subplots(2, NUM_VIEWS, figsize=(4 * NUM_VIEWS, 8))
for i in range(NUM_VIEWS):
    axes[0, i].imshow(target_silhouettes[i].detach().cpu(), cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"Target · View {i + 1}")
    axes[0, i].axis("off")
    axes[1, i].imshow(final_silhouettes[i].detach().cpu(), cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Optimized · View {i + 1}")
    axes[1, i].axis("off")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "final_silhouette_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

# 2) 损失曲线
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history_total, label="Total Loss")
ax.plot(history_silhouette, label="Silhouette MSE")
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("Optimization Loss Curve")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "loss_curve.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

# 3) 优化过程 GIF
imageio.mimsave(
    RESULTS_DIR / "optimization_progress.gif",
    gif_frames,
    duration=0.45,
    loop=0,
)

# 4) 导出最终网格 OBJ
save_obj(
    RESULTS_DIR / "final_mesh.obj",
    final_mesh.verts_packed().detach().cpu(),
    final_mesh.faces_packed().detach().cpu(),
)

# 5) 保存参数和最终损失
metrics_text = (
    "Experiment: Differentiable Rendering - Cow Silhouette Fitting\n"
    f"Device: {device}\n"
    "Target model: cow.obj\n"
    f"Views: {NUM_VIEWS}\n"
    f"Image size: {IMAGE_SIZE} x {IMAGE_SIZE}\n"
    f"Iterations: {NUM_ITERS}\n"
    "Optimizer: Adam(lr=0.08)\n"
    f"Weights: W_LAP={W_LAP}, W_EDGE={W_EDGE}, W_NORMAL={W_NORMAL}\n"
    f"Final total loss: {history_total[-1]:.8f}\n"
    f"Final silhouette MSE: {history_silhouette[-1]:.8f}\n"
)
(RESULTS_DIR / "metrics.txt").write_text(metrics_text, encoding="utf-8")

# 6) 打包输出结果
archive_path = shutil.make_archive("work06_results", "zip", RESULTS_DIR)
print(metrics_text)
print("结果已保存到：", RESULTS_DIR.resolve())
print("结果压缩包：", Path(archive_path).resolve())

In [ ]:
# 自动下载结果压缩包。下载后将其解压，并把其中所有文件复制到 GitHub 项目 work06/results/ 目录。
from google.colab import files
files.download("work06_results.zip")